# Reaction-time models with permutation test

Two models are compared. They differ only in what goes in the predictor column:

| model | predictor for a trial |
|---|---|
| **continuous** | mean decoding accuracy of the letters in the string |
| **binary** | fraction of the string's letters that are *significantly* decoded  |

In the permutation test, the 15 letter values are shuffled across letters, every trial's predictor is
recomputed, and the model is refitted. Because each null replicate is produced by the same model,
whatever inflation the model has appears in the null too.

Probe identity is tested as a whole factor with a likelihood-ratio test.

In [1]:
import itertools, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

plt.rcParams.update({'font.size': 12})
LETTERS = ['B','C','D','F','G','H','K','L','N','P','R','S','T','V','Z']
L_IDX   = {l: i for i, l in enumerate(LETTERS)}

# ---- configuration -----------------------------------------------------------
N_PERM_CONT  = 5000     # Monte Carlo draws for the continuous model
EXACT_BINARY = True     
SEED         = 0
ZSCORE_RT    = False    # also z-score RT within subject (coefficients then in within-subject SD units)

## 1. Load data


In [2]:
info    = np.load('../Data/trial_corrInfo.npy', allow_pickle=True)
letters = np.load('../Data/letters_sep.npy',    allow_pickle=True)
probes  = np.load('../Data/probes_sep.npy',     allow_pickle=True)
subjs   = np.load('../Data/subjs_sep.npy',      allow_pickle=True)

subj    = np.concatenate(subjs, axis=0)
rawL    = [list(s) for s in np.concatenate(letters, axis=0)]
rawP    = [list(s) for s in np.concatenate(probes,  axis=0)]

infoALL = np.concatenate(info)
n_all, n_ok = len(infoALL), int((infoALL[:, 1] == 1).sum())
infoALL = infoALL[infoALL[:, 1] == 1]

assert len(rawL) == n_ok == len(subj)
print(f'trials: {n_all} total -> {n_ok} correct;  participants: {sorted(np.unique(subj))}')

trials: 2849 total -> 2613 correct;  participants: [40, 43, 44, 45, 47, 53, 57, 58, 61, 63, 64]


## 2. Decoding accuracy and significance per letter

In [3]:
T1, T2 = 2, 16

def load_decoding(n_splits=50):
    acc = [np.array([np.diag(np.load(f'../../FC_letter_decoding_updated/v2/Results/single_letter_50_{k}.npy')[i])
                     for i in range(15)]) for k in range(n_splits)]
    acc = np.array(acc)
    return np.median(np.array([np.cumsum(acc[:, i, T1:T2], axis=-1)[:, -1] / (T2 - T1)
                               for i in range(15)]), axis=1)

DEC = load_decoding(); print('decoding accuracies loaded from ../Results/')

SIG_LETTERS = ['F','B','G','C','V','D','N','R']     # FDR p<.05, see 2c_letter_decoding 
sig = np.array([l in SIG_LETTERS for l in LETTERS])
print(pd.DataFrame({'decoding_acc': DEC.round(4), 'significant': sig},
                   index=LETTERS).T.to_string())

decoding accuracies loaded from ../Results/
                   B      C       D       F       G      H       K       L      N       P     R       S       T       V       Z
decoding_acc  0.5725  0.555  0.5461  0.5964  0.5586    0.5  0.5214  0.4896  0.545  0.4782  0.54  0.4954  0.5082  0.5468  0.5054
significant     True   True    True    True    True  False   False   False   True   False  True   False   False    True   False


## 3. Phonological covariates

In [4]:
FEAT = np.array([
 [1,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0],[0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0],
 [1,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0],[0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,1,0],
 [1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0],[0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0],
 [0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0],[1,0,0,1,0,0,0,0,0,0,1,0,1,0,0,1,0],
 [1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0],[0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0],
 [1,0,0,1,0,0,0,0,0,0,0,1,1,0,0,1,0],[0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0],
 [0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0],[0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1],
 [0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0]], dtype=float)
_N  = FEAT / np.linalg.norm(FEAT, axis=1, keepdims=True)
COS = _N @ _N.T

def mean_pairwise_cosine(ls):
    idx = [L_IDX[l] for l in ls]
    if len(idx) < 2: return np.nan
    return np.mean([COS[idx[a], idx[b]]
                    for a in range(len(idx)) for b in range(a+1, len(idx))])

def probe_string_cosine(p, ls):
    o = [l for l in ls if l != p]
    return np.mean([COS[L_IDX[p], L_IDX[l]] for l in o]) if o else np.nan

## 4. Trial dataframe

In [5]:
rows = []
for i in range(len(infoALL)):
    raw_s, raw_p = rawL[i], rawP[i]
    ls = [c for c in raw_s if c in L_IDX]
    ps = [c for c in raw_p if c in L_IDX]
    p  = ps[0] if len(ps) else None
    inf = infoALL[i]
    r = {'rt': inf[4], 'trial_type': 'IN' if int(inf[3]) == 51 else 'OUT',
         'size': int(inf[0]), 'subject': subj[i], 'probe': p,
         'string': ''.join(ls),
         'phon_sim': mean_pairwise_cosine(ls),
         'probe_phon_sim': probe_string_cosine(p, ls) if p else np.nan}
    for l in LETTERS:
        r['has_' + l] = float(l in ls)
    rows.append(r)

df = (pd.DataFrame(rows)
      .dropna(subset=['rt','probe','phon_sim','probe_phon_sim'])
      .reset_index(drop=True))
print(f'n = {len(df)} trials, {df.subject.nunique()} participants')

n = 2585 trials, 11 participants


## 5. The two models

Identical apart from the predictor column. All continuous predictors are **z-scored within each subject**
(subject mean subtracted, divided by the subject's SD), so the effects reflect within-subject variation.
Because the decoding predictor is rebuilt on every permutation, it is re-z-scored within subject on every
permutation as well. Set `ZSCORE_RT = True` to also z-score RT within subject.

```
rt ~ predicted_z + phon_sim_z + probe_phon_sim_z
     + C(trial_type) + C(size) + C(probe) + (1 | subject)
```

In [6]:
BASE = ('rt ~ predicted_z + phon_sim_z + probe_phon_sim_z '
        '+ C(trial_type) + C(size) + C(probe)')

def zscore_within(x, codes):
    """z-score x separately within each subject (codes = integer subject index per trial)."""
    x   = np.asarray(x, dtype=float)
    n   = np.bincount(codes)
    mu  = np.bincount(codes, weights=x) / n
    var = np.bincount(codes, weights=x**2) / n - mu**2
    sd  = np.sqrt(np.clip(var * n / np.maximum(n - 1, 1), 0, None))   # sample SD (ddof=1)
    sd  = np.where(sd > 0, sd, np.nan)
    z   = (x - mu[codes]) / sd[codes]
    return np.nan_to_num(z, nan=0.0)             # a subject with no variance contributes 0

def add_predictor(d, Hn, values):
    """Build the per-trial decoding predictor from 15 letter values and z-score it within subject."""
    pv = Hn @ np.asarray(values, dtype=float)
    d['predicted_z'] = zscore_within(pv, d['subj_code'].to_numpy())
    return d

def prep(data):
    d  = data.copy().reset_index(drop=True)
    H  = d[['has_' + l for l in LETTERS]].to_numpy()
    Hn = H / H.sum(1, keepdims=True)                 # row-normalised -> mean over the string
    codes = pd.factorize(d['subject'])[0]
    d['subj_code'] = codes
    for c in ['phon_sim', 'probe_phon_sim']:
        d[c + '_z'] = zscore_within(d[c], codes)
    if ZSCORE_RT:
        d['rt_raw'] = d['rt']
        d['rt'] = zscore_within(d['rt'], codes)
    d['trial_type'] = pd.Categorical(d.trial_type, ['OUT', 'IN'])
    d['size']       = pd.Categorical(d['size'], sorted(d['size'].unique()))
    d['probe']      = pd.Categorical(d.probe, ['G'] + [l for l in LETTERS if l != 'G'])
    return d, Hn

def fit_z(d, Hn, values):
    """values: length-15, one number per letter. Returns (beta, z, nominal p)."""
    dd = add_predictor(d.copy(), Hn, values)
    r  = smf.mixedlm(BASE, dd, groups=dd['subject'], re_formula='1').fit(reml=False)
    b  = r.fe_params['predicted_z']
    return b, b / r.bse_fe['predicted_z'], r.pvalues['predicted_z']

# quick check that both models fit
for nm, v in [('continuous', DEC), ('binary', sig.astype(float))]:
    d, Hn = prep(df)
    b, z, p = fit_z(d, Hn, v)
    unit = 'SD' if ZSCORE_RT else 's'
    print(f'{nm:11s} beta = {b:+.4f} {unit}/SD   z = {z:+.3f}   nominal p = {p:.5f}')

continuous  beta = -0.0339 s/SD   z = -2.397   nominal p = 0.01654
binary      beta = -0.0483 s/SD   z = -3.401   nominal p = 0.00067


## 6. Permutation

In [7]:
ALL_SPLITS = [np.isin(np.arange(15), s)
              for s in itertools.combinations(range(15), len(SIG_LETTERS))]

def permute(data, values, kind, label):
    d, Hn = prep(data)
    _, z_obs, p_nom = fit_z(d, Hn, values)

    rng = np.random.default_rng(SEED)
    if kind == 'binary' and EXACT_BINARY:
        draws, exact = [m.astype(float) for m in ALL_SPLITS], True
    elif kind == 'binary':
        draws = [rng.permutation(sig).astype(float) for _ in range(N_PERM_CONT)]; exact = False
    else:
        draws = [rng.permutation(values) for _ in range(N_PERM_CONT)]; exact = False

    t0, zn = time.time(), np.empty(len(draws))
    for k, v in enumerate(draws):
        zn[k] = fit_z(d, Hn, v)[1]
        if k == 24:
            print(f'  [{label}] {(time.time()-t0)/25:.3f}s/fit, '
                  f'ETA {len(draws)*(time.time()-t0)/25/60:.1f} min', flush=True)

    p2 = np.mean(np.abs(zn) >= abs(z_obs) - 1e-12)
    p1 = np.mean(zn <= z_obs + 1e-12)
    return {'label': label, 'n': len(d), 'z_obs': z_obs, 'p_nominal': p_nom,
            'n_perm': len(zn), 'exact': exact, 'null': zn,
            'null_sd': zn.std(), 'p_perm_2s': p2, 'p_perm_1s': p1,
            'n_extreme': int(np.sum(np.abs(zn) >= abs(z_obs) - 1e-12))}

In [8]:
runs = []
for kind, values in [('continuous', DEC), ('binary', sig.astype(float))]:
    print(f'running {kind} ...', flush=True)
    runs.append(permute(df, values, 'binary' if kind.startswith('binary') else 'continuous',
                        kind))
print('done')

running continuous ...
  [continuous] 0.135s/fit, ETA 11.3 min
running binary ...
  [binary] 0.148s/fit, ETA 15.8 min
done


## 7. Results

In [9]:
runs[0]

{'label': 'continuous',
 'n': 2585,
 'z_obs': -2.3967830309507234,
 'p_nominal': 0.016539714275600994,
 'n_perm': 5000,
 'exact': False,
 'null': array([-0.38782424,  2.7479118 , -0.26219725, ...,  1.96643375,
        -0.67256992,  0.37142968]),
 'null_sd': 1.5077601583700055,
 'p_perm_2s': 0.1154,
 'p_perm_1s': 0.0544,
 'n_extreme': 577}

In [10]:
summary = pd.DataFrame([{
    'model':          r['label'],
    'n trials':       r['n'],
    'z observed':     round(r['z_obs'], 3),
    'p nominal':      f"{r['p_nominal']:.5f}",
    'null SD':        round(r['null_sd'], 3),
    'n perm':         r['n_perm'],
    'exact':          r['exact'],
    'extreme':        r['n_extreme'],
    'p permutation':  round(r['p_perm_2s'], 4)} for r in runs])
print(summary.to_string(index=False))

import os
os.makedirs('../Results', exist_ok=True)
summary.to_csv('../Results/permutation_summary.csv', index=False)
np.save('../Results/permutation_nulls.npy',
        np.array([r['null'] for r in runs], dtype=object), allow_pickle=True)

     model  n trials  z observed p nominal  null SD  n perm  exact  extreme  p permutation
continuous      2585      -2.397   0.01654    1.508    5000  False      577         0.1154
    binary      2585      -3.401   0.00067    1.496    6435   True       54         0.0084


## 8. Full fixed-effects tables

In [11]:
def sig3(x):
    """Format a number to 3 significant digits."""
    if x == 0 or pd.isna(x):
        return '0.00'
    return f'{x:.3g}'

def fit_full(values):
    d, Hn = prep(df)
    d = add_predictor(d, Hn, values)
    return smf.mixedlm(BASE, d, groups=d['subject'], re_formula='1').fit(reml=False)

RENAME = {'Intercept':        'Intercept',
          'phon_sim_z':       'Phonological similarity (string)',
          'probe_phon_sim_z': 'Phonological similarity (probe-string)',
          'C(trial_type)[T.IN]': 'IN vs. OUT',
          'C(size)[T.6]':     'Set size 6 vs. 4',
          'C(size)[T.8]':     'Set size 8 vs. 4'}

def clean(term, key_name):
    if term == 'predicted_z':
        return key_name
    if term in RENAME:
        return RENAME[term]
    if term.startswith('C(probe)[T.'):
        return term[len('C(probe)[T.'):-1] + ' vs. G'
    return term

def coef_table(r, key_name, perm_p):
    idx = r.fe_params.index
    ci  = r.conf_int().loc[idx]
    z_vals = r.fe_params / r.bse_fe
    t = pd.DataFrame({
        'Coefficient': [sig3(v) for v in r.fe_params.loc[idx]],
        '95% CI': [f'[{sig3(lo)}, {sig3(hi)}]' for lo, hi in zip(ci.iloc[:, 0], ci.iloc[:, 1])],
        'z': [sig3(v) for v in z_vals.loc[idx]],
        'p (model)': [sig3(p) for p in r.pvalues.loc[idx]],
        'p (permutation)': ['' for _ in idx]},
        index=[clean(t_, key_name) for t_ in idx])
    t.loc[key_name, 'p (permutation)'] = sig3(perm_p)

    priority = ['Intercept', key_name,
                'Phonological similarity (string)',
                'Phonological similarity (probe-string)',
                'IN vs. OUT', 'Set size 6 vs. 4', 'Set size 8 vs. 4']
    order = [x for x in priority if x in t.index] + \
            [x for x in t.index if x not in priority]
    return t.loc[order]

tables = {}
for r, values, key in [(runs[0], DEC, 'Decoding accuracy (continuous)'),
                       (runs[1], sig.astype(float), 'Decoding accuracy (binary)')]:
    m = fit_full(values)
    tab = coef_table(m, key, r['p_perm_2s'])
    tables[r['label']] = tab
    print(f"===== {r['label']}  (n = {r['n']}, {df.subject.nunique()} participants) =====")
    print(tab.to_string())
    print()
    tab.to_excel(f"../Results/fixed_effects_{'binary' if 'binary' in r['label'] else 'continuous'}.xlsx")

===== continuous  (n = 2585, 11 participants) =====
                                       Coefficient               95% CI       z p (model) p (permutation)
Intercept                                     1.45         [1.21, 1.68]    12.2  4.48e-34                
Decoding accuracy (continuous)             -0.0339  [-0.0617, -0.00619]    -2.4    0.0165           0.115
Phonological similarity (string)           -0.0437   [-0.0742, -0.0132]   -2.81   0.00501                
Phonological similarity (probe-string)       0.075      [0.0345, 0.115]    3.63  0.000285                
IN vs. OUT                                  -0.146    [-0.201, -0.0901]   -5.14   2.7e-07                
Set size 6 vs. 4                             0.217       [0.152, 0.283]    6.53  6.78e-11                
Set size 8 vs. 4                             0.341        [0.271, 0.41]     9.6  8.03e-22                
B vs. G                                     0.0995     [-0.0552, 0.254]    1.26     0.207           

## 9. Likelihood-ratio test for probe identity

In [12]:
from scipy import stats

NO_PROBE = BASE.replace(' + C(probe)', '')
assert NO_PROBE != BASE, 'C(probe) not found in the model formula'


def fit_model(values, formula):
    """Fit the RT mixed model by maximum likelihood, using `values` to build the decoding predictor."""
    d, Hn = prep(df)
    d = add_predictor(d, Hn, values)
    model = smf.mixedlm(formula, d, groups=d['subject'], re_formula='1')
    return model.fit(reml=False)


def probe_lrt(label, values):
    """Likelihood-ratio test for probe identity: full model vs. model without C(probe)."""
    full = fit_model(values, BASE)
    reduced = fit_model(values, NO_PROBE)

    # The LRT is only valid if both fits use the same trials
    assert full.nobs == reduced.nobs
    if not (full.converged and reduced.converged):
        print(f'  WARNING [{label}]: a fit did not converge; treat the LRT with caution')

    n_df = len(full.fe_params) - len(reduced.fe_params)   # number of probe dummy columns
    chi2 = 2 * (full.llf - reduced.llf)
    p = stats.chi2.sf(chi2, n_df)

    return {
        'model': label,
        'term': 'Probe identity',
        'df': n_df,
        'logLik full': round(full.llf, 2),
        'logLik reduced': round(reduced.llf, 2),
        'LRT chi2': float(sig3(chi2)),
        'p (LRT)': float(sig3(p)),
    }


predictors = {
    'continuous': DEC,
    'binary': sig.astype(float),
}
rows = [probe_lrt(label, values) for label, values in predictors.items()]

results = pd.DataFrame(rows)
print(results.to_string(index=False))
results.to_csv('../Results/probe_identity_lrt.csv', index=False)

# Add the test as a second sheet to each fixed-effects workbook
for row in rows:
    path = f"../Results/fixed_effects_{row['model']}.xlsx"
    sheet = pd.DataFrame([row]).drop(columns='model').set_index('term')
    with pd.ExcelWriter(path, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        sheet.to_excel(writer, sheet_name='probe omnibus')

     model           term  df  logLik full  logLik reduced  LRT chi2  p (LRT)
continuous Probe identity  14     -2815.20        -2834.45      38.5 0.000435
    binary Probe identity  14     -2812.29        -2831.64      38.7 0.000406


## 10. Serial position as sole predictor (IN trials only)

```
rt ~ serial_pos_z + C(size) + (1 | subject)
```

No phonological or decoding predictors, so this isolates the positional effect. The probe's position in the
string is normalised to [0, 1] (first = 0, last = 1) and then **z-scored within subject**, like the other
continuous predictors. RT is z-scored within subject only if `ZSCORE_RT = True`.

In [14]:
df_pos = df[df['trial_type'] == 'IN'].copy().reset_index(drop=True)

# normalised serial position of the probe in the string: 0 = first, 1 = last
def serial_pos(row):
    s = row['string']
    if row['probe'] not in s or row['size'] < 2:
        return np.nan
    return s.index(row['probe']) / (row['size'] - 1)

df_pos['serial_pos_norm'] = df_pos.apply(serial_pos, axis=1)

df_pos = df_pos.dropna(subset=['serial_pos_norm', 'rt']).reset_index(drop=True)

# z-score within subject (subtracting the grand mean first is unnecessary: the subject mean removes it)
codes = pd.factorize(df_pos['subject'])[0]
df_pos['serial_pos_z'] = zscore_within(df_pos['serial_pos_norm'], codes)
if ZSCORE_RT:
    df_pos['rt_raw'] = df_pos['rt']
    df_pos['rt'] = zscore_within(df_pos['rt'], codes)

# set size as a factor, reference = 4
df_pos['size'] = pd.Categorical(df_pos['size'], sorted(df_pos['size'].unique()))

print(f'n = {len(df_pos)} IN trials, {df_pos.subject.nunique()} participants')
model_pos_only = smf.mixedlm('rt ~ serial_pos_z + C(size)', df_pos,
                             groups=df_pos['subject'], re_formula='1')
res_pos_only = model_pos_only.fit(reml=False)
if not res_pos_only.converged:
    print('WARNING: model did not converge')
print(res_pos_only.summary())

n = 1191 IN trials, 11 participants
      count      mean       std
size                           
4     519.0  0.490045  0.366230
6     376.0  0.519149  0.336342
8     296.0  0.481178  0.345036
         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: rt        
No. Observations: 1191    Method:             ML        
No. Groups:       11      Scale:              0.4712    
Min. group size:  75      Log-Likelihood:     -1258.6019
Max. group size:  149     Converged:          Yes       
Mean group size:  108.3                                 
--------------------------------------------------------
               Coef. Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept      1.398    0.095 14.732 0.000  1.212  1.584
C(size)[T.6]   0.170    0.047  3.630 0.000  0.078  0.261
C(size)[T.8]   0.219    0.050  4.356 0.000  0.120  0.317
serial_pos_z   0.006    0.020  0.310 0.757 -0.033  0.045
Group Var      0